In [ ]:
from main import sheet_processor
import logging
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import json
import os
from utils.processing import DataSampler, create_report


# How to set log level to debug
logging.basicConfig(level=logging.DEBUG)

logger = logging.getLogger(__name__)

## Model Selection

In this section, we specify which models the system will use. To simplify the initial setup and testing workflow, we currently use a single selected model across all processing components: **PII Detection, PII Reflection, Non-PII Detection, and ReadMe Detection**. This approach allows us to validate the end-to-end pipeline without introducing variability from multiple model behaviors.

At the moment, only two models are deployed through the Azure service: **GPT-4.1 Mini** and **GPT-4.1 Nano**. These models are the available choices for experimentation and integration. As the project evolves, additional models can be added or different models can be assigned to specific components for more specialized behavior.


In [ ]:
MODEL = 'gpt-4.1-nano'

ALLOWED_MODELS = ['gpt-4.1-mini', 'gpt-4.1-nano']

if MODEL not in ALLOWED_MODELS:
    raise ValueError(f'Invalid model: {MODEL}. Please use one of the following: {", ".join(ALLOWED_MODELS)}')

# Dataset Selection

You can select a dataset in one of two ways:

1. **Using a download URL**: Provide the URL to a CSV, XLS, or XLSX file.
2. **Using a local file**: Choose a file from the `research/data` folder in the project.

This flexibility allows you to work with both online datasets and local files for testing and analysis.


In [ ]:
file_path = 'research/data/'  # This is a local test file

# DataSampler

The `DataSampler` class is used to extract a subset of rows from the original dataset.  
This sampling helps to:

- **Reduce memory usage** when working with large datasets.
- **Increase processing speed** during classification and analysis.

By working on a smaller, representative portion of the data, we can efficiently test the pipeline and classifiers without loading the entire dataset.


In [ ]:
# Start by making empty reports for each file as groundtruth to fill in later

sampler = DataSampler()

with open('/Users/liangtelkamp/Documents/GitHub/hdx-ssd-pipeline/data/isps.json', 'r') as f:
    isp = json.load(f)

isp = isp['default']

for file in os.listdir('research/data'):
    file_path = f'research/data/{file}'
    # If file already exists, skip
    if os.path.exists(f'research/results/test_results/groundtruth/{file}.json'):
        logger.info(f'File {file} already exists. Skipping.')
        continue
    try:
        sdd_report = create_report(file_path)
        with open(f'research/results/test_results/groundtruth/{file}.json', 'w') as f:
            json.dump(sdd_report, f, indent=4)
        logger.info(f'Report saved to research/results/test_results/groundtruth/{file}.json')
    except Exception as e:
        logger.warning(f'Error processing file: {e}')
        continue

[84326 - 8687722688] 2025-12-11 22:35:26,056 WARNI [__main__:22] Error processing file: bgd_dataset_joint-msna_refugee_september-2019.xlsx
[84326 - 8687722688] 2025-12-11 22:35:26,058 INFO  [__main__:14] File afghanistan_june_protection.csv already exists. Skipping.
[84326 - 8687722688] 2025-12-11 22:35:26,206 WARNI [__main__:22] Error processing file: ujana-coffee-project-database-2019-2024.xlsx
[84326 - 8687722688] 2025-12-11 22:35:26,206 INFO  [__main__:14] File sira_household.xlsx already exists. Skipping.
[84326 - 8687722688] 2025-12-11 22:35:26,207 INFO  [__main__:14] File afghanistan_june_health.csv already exists. Skipping.
[84326 - 8687722688] 2025-12-11 22:35:26,207 INFO  [__main__:14] File isna_analysis_dataset_08_10_2024-final-clean-dataset-hdx.xlsx already exists. Skipping.
[84326 - 8687722688] 2025-12-11 22:35:26,208 INFO  [__main__:14] File ACF_Pastoral_Sentinels.xlsx already exists. Skipping.
[84326 - 8687722688] 2025-12-11 22:35:26,208 INFO  [__main__:14] File acf_donn

# Processing Each Sheet Individually

For each sheet in the dataset, we perform the following processing steps:

1. **PII Detection** – Identify columns containing personally identifiable information.
2. **PII Reflection Detection** – Detect columns that might indirectly reveal PII.
3. **Non-PII Detection** – Classify remaining columns that do not contain sensitive information.

**Special Case:**  
If a sheet is named `readme`, `instructions`, or `metadata`, we skip the column-level classification and instead perform a **simple ReadMe scan** to extract relevant information from the documentation.


In [7]:
# Start making sdd reports for a model
MODEL = 'gpt-4.1-nano'
# Start by making empty reports for each file as groundtruth to fill in later

sampler = DataSampler()

with open('/Users/liangtelkamp/Documents/GitHub/hdx-ssd-pipeline/data/isps.json', 'r') as f:
    isp = json.load(f)

isp = isp['default']

for file in os.listdir('research/data'):
    file_path = f'research/data/{file}'
    # Check if file already exists
    output_path = f'research/results/test_results/{MODEL}/{file}.json'
    if os.path.exists(output_path):
        logger.info(f'File {file} already exists, loading existing report')
        # Load the existing report
        with open(output_path, 'r') as f:
            sdd_report = json.load(f)
        logger.info(f'Report loaded from {output_path}')
    else:
        # Check if file is already processed
        if not os.path.exists(f'research/results/test_results/groundtruth/{file}.json'):
            logger.warning(f'File {file} not found in groundtruth, skipping')
            continue
        else:
            with open(f'research/results/test_results/groundtruth/{file}.json', 'r') as f:
                logger.info(f'File {file} found in groundtruth, loading')
                sdd_report = json.load(f)
    reports = []
    for sheet in sdd_report:
        sdd_report = sheet_processor(sheet, isp, MODEL)
        reports.append(sdd_report)
    with open(output_path, 'w') as f:
        json.dump(reports, f, indent=4)
    logger.info(f'Report saved to {output_path}')

[84562 - 8687722688] 2025-12-11 22:45:24,626 WARNI [__main__:26] File bgd_dataset_joint-msna_refugee_september-2019.xlsx not found in groundtruth, skipping
[84562 - 8687722688] 2025-12-11 22:45:24,627 INFO  [__main__:17] File afghanistan_june_protection.csv already exists, loading existing report
SDD Report: [{'resource_id': None, 'file_name': 'research/data/afghanistan_june_protection.csv', 'file_url': None, 'sheet_name': 'sheet1', 'processing_timestamp': '2025-12-11 22:35:31', 'processing_success': True, 'n_records': 199, 'n_columns': 46, 'completion_tokens': 122, 'prompt_tokens': 25048, 'columns': [{'column_name': 'observed_time', 'sample_values': ['2025-06-11T14:45:14.114Z', '2025-06-02T04:03:30.526Z', '2025-06-02T04:20:02.994Z', '2025-06-02T04:34:17.462Z', '2025-06-02T04:46:41.049Z'], 'pii': {'entity_type': 'None', 'sensitive': False}}, {'column_name': 'form_version', 'sample_values': ['8', '8', '8', '8', '8'], 'pii': {'entity_type': 'None', 'sensitive': False}}, {'column_name': '

Reflecting on PII sensitivity: 100%|██████████| 13/13 [00:01<00:00,  6.68it/s]


Non-PII classification: {'resource_id': None, 'file_name': 'research/data/sira_household.xlsx', 'file_url': None, 'sheet_name': 'Sheet1', 'processing_timestamp': '2025-12-11 22:29:50', 'processing_success': True, 'n_records': 999, 'n_columns': 13, 'completion_tokens': 252, 'prompt_tokens': 5867, 'columns': [{'column_name': 'hhID', 'sample_values': ['caap5lrlvgduqk05', 'ckq4r1xlvmbev9y2q', 'ct42s3klvmbaizj1f', 'c608zrtlvmbbtur1r', 'c5lst8nlvmbvgm32'], 'pii': {'entity_type': 'None', 'sensitive': False}}, {'column_name': 'Parent_orgENUM', 'sample_values': ['FAMOD', 'FAMOD', 'FAMOD', 'FAMOD', 'FAMOD'], 'pii': {'entity_type': 'None', 'sensitive': False}}, {'column_name': 'Parent_District', 'sample_values': ['Metuge', 'Metuge', 'Metuge', 'Metuge', 'Metuge'], 'pii': {'entity_type': 'None', 'sensitive': False}}, {'column_name': 'Parent_Subdistrict', 'sample_values': ['Metuge', 'Metuge', 'Metuge', 'Metuge', 'Metuge'], 'pii': {'entity_type': 'None', 'sensitive': False}}, {'column_name': 'Parent_

Reflecting on PII sensitivity: 100%|██████████| 47/47 [00:03<00:00, 11.96it/s]


[84562 - 8687722688] 2025-12-11 22:46:06,455 INFO  [__main__:38] Report saved to research/results/test_results/gpt-4.1-nano/afghanistan_june_health.csv.json
[84562 - 8687722688] 2025-12-11 22:46:06,456 INFO  [__main__:30] File isna_analysis_dataset_08_10_2024-final-clean-dataset-hdx.xlsx found in groundtruth, loading
SDD Report: {'resource_id': None, 'file_name': 'research/data/isna_analysis_dataset_08_10_2024-final-clean-dataset-hdx.xlsx', 'file_url': None, 'sheet_name': 'READ_ME', 'processing_timestamp': '2025-12-11 22:29:52', 'processing_success': True, 'n_records': 14, 'n_columns': 3, 'completion_tokens': 0, 'prompt_tokens': 0, 'columns': [{'column_name': 'ISNA South Sudan 2024 | ISNA South Sudan 2024 | ISNA South Sudan 2024 | Items  | Project Background  | Primary data collection time period | Methodology: \nOverview', 'sample_values': ['Population Groups Assessed', 'Reporting and level of representativeness', '\nGeographic coverage\n', '\nScope\n', 'Contacts'], 'pii': {'entity_ty

...

Non-PII classification: {'resource_id': None, 'file_name': 'research/data/isna_analysis_dataset_08_10_2024-final-clean-dataset-hdx.xlsx', 'file_url': None, 'sheet_name': 'Analysis - County', 'processing_timestamp': '2025-12-11 22:29:52', 'processing_success': True, 'n_records': 999, 'n_columns': 12, 'completion_tokens': 265, 'prompt_tokens': 4797, 'columns': [{'column_name': 'Sector/Cluster', 'sample_values': ['Demographics', 'Demographics', 'Demographics', 'Demographics', 'Demographics'], 'pii': {'entity_type': 'None', 'sensitive': False}}, {'column_name': 'Indicator', 'sample_values': ['Average age of HoHH', '% of responses given by HoHH', '% of responses given by HoHH', '% of responses given by HoHH', '% of responses given by HoHH'], 'pii': {'entity_type': 'None', 'sensitive': False}}, {'column_name': 'Survey.Question', 'sample_values': ['Age of household Head:', 'Is the respondent the head of household?', 'Is the respondent the head of household?', 'Is the respondent the head of ho

Reflecting on PII sensitivity: 100%|██████████| 12/12 [00:01<00:00,  7.54it/s]


Non-PII classification: {'resource_id': None, 'file_name': 'research/data/isna_analysis_dataset_08_10_2024-final-clean-dataset-hdx.xlsx', 'file_url': None, 'sheet_name': 'County by Population Group', 'processing_timestamp': '2025-12-11 22:29:52', 'processing_success': True, 'n_records': 999, 'n_columns': 12, 'completion_tokens': 319, 'prompt_tokens': 5898, 'columns': [{'column_name': 'Sector/Cluster', 'sample_values': ['Demographics', 'Demographics', 'Demographics', 'Demographics', 'Demographics'], 'pii': {'entity_type': 'None', 'sensitive': False}}, {'column_name': 'Indicator', 'sample_values': ['Average age of HoHH', '% of HoHH by sex', '% of HoHH by sex', '% of HoHH by sex', '% of HoHH by sex'], 'pii': {'entity_type': 'None', 'sensitive': False}}, {'column_name': 'Survey.Question', 'sample_values': ['Age of household Head:', 'Sex of household Head', 'Sex of household Head', 'Sex of household Head', 'Sex of household Head'], 'pii': {'entity_type': 'None', 'sensitive': False}}, {'colu

Classifying PII:  10%|█         | 50/478 [00:24<03:32,  2.01it/s]


KeyboardInterrupt: 

# Save


# Evaluation


In [6]:
def compare_pii_columns(gt_reports, pred_reports):
    """Compare PII sensitivity for all columns across sheets."""
    records = []
    for gt, pred in zip(gt_reports, pred_reports):
        gt_cols = {c['column_name']: c['pii']['sensitive'] for c in gt['columns']}
        pred_cols = {c['column_name']: c['pii']['sensitive'] for c in pred['columns']}
        for col_name in gt_cols:
            records.append({'column_name': col_name, 'true': gt_cols[col_name], 'pred': pred_cols.get(col_name, False)})
    return pd.DataFrame(records)


def compare_pii_table_level(gt_reports, pred_reports):
    """Compare PII sensitivity at the table level."""
    records = []
    for gt, pred in zip(gt_reports, pred_reports):
        records.append({'true': gt.get('pii_sensitive', False), 'pred': pred.get('pii_sensitive', False)})
    return pd.DataFrame(records)


def compare_non_pii_table_level(gt_reports, pred_reports):
    """Compare non-PII sensitivity at the table level."""
    records = []
    for gt, pred in zip(gt_reports, pred_reports):
        records.append({'true': gt.get('non_pii_sensitive', False), 'pred': pred.get('non_pii_sensitive', False)})
    return pd.DataFrame(records)


def calculate_metrics(df: pd.DataFrame):
    """Compute accuracy, precision, recall, and F1 score."""
    return {
        'accuracy': accuracy_score(df['true'], df['pred']),
        'precision': precision_score(df['true'], df['pred'], zero_division=0),
        'recall': recall_score(df['true'], df['pred'], zero_division=0),
        'f1': f1_score(df['true'], df['pred'], zero_division=0),
    }

In [7]:
filename = 'data.xlsx'

# Read the groundtruth and predictions
with open(f'research/results/test_results/groundtruth/{filename}.json', 'r') as f:
    groundtruth = json.load(f)
with open(f'research/results/test_results/gpt-4.1-nano/{filename}.json', 'r') as f:
    predictions = json.load(f)

# Calculate metrics
metrics = {
    filename: {
        'pii_columns': calculate_metrics(compare_pii_columns(groundtruth, predictions)),
        'pii_table_level': calculate_metrics(compare_pii_table_level(groundtruth, predictions)),
        'non_pii_table_level': calculate_metrics(compare_non_pii_table_level(groundtruth, predictions)),
    }
}
metrics

FileNotFoundError: [Errno 2] No such file or directory: 'research/results/test_results/groundtruth/data.xlsx.json'